# PyTorch Fundamentals: Follow-Along Notebook

This is a live-coding notebook. Start with the TODO cells incomplete and write the missing code together during class.

## Core path

The class is successful when we can:

1. create and reshape tensors,
2. calculate a gradient with autograd,
3. define an `nn.Module`,
4. load Fashion-MNIST mini-batches,
5. perform one parameter update,
6. train and evaluate the model.

## Stretch path

If time remains, we will:

7. plot learning curves,
8. inspect confident mistakes,
9. save and reload the model.

Do not rush the core path to complete every section. Understanding one complete training loop is the priority.

> **Important:** do not use **Run all** at the beginning. TODO cells are intentionally incomplete and should be filled sequentially.

## Suggested three-hour pace

| Time | Target |
|---|---|
| 0–15 min | Setup and device check |
| 15–40 min | Tensors, shapes, and matrix multiplication |
| 40–60 min | Autograd and gradient clearing |
| 60–85 min | Build and test `FashionMLP` |
| 85–105 min | Fashion-MNIST and DataLoaders |
| 105–125 min | One complete parameter update |
| 125–135 min | Break |
| 135–170 min | Training loop, evaluation loop, and test result |
| Remaining time | Learning curves, mistakes, or saving the model |

## 0. Colab setup

Run this cell first. Google Colab normally includes PyTorch, NumPy, and Matplotlib, so the helper installs only missing packages.

If packages are installed, wait for the cell to finish before running the imports.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing:", ", ".join(missing_packages))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )
    print("Installation complete.")
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [ ]:
%matplotlib inline

from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.style.use("seaborn-v0_8-whitegrid")
torch.set_printoptions(precision=4, sci_mode=False)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


### TODO 1: Select the available device

Use this priority:

1. CUDA GPU
2. Apple MPS
3. CPU

**Checkpoint:** printing `device` should show `cuda`, `mps`, or `cpu`.

In [ ]:
# TODO: Select CUDA, MPS, or CPU and store it in `device`.


print("Selected device:", device)

# Core 1: Tensors and shapes

## 1. Create three tensors

Create:

- a $2\times3$ integer tensor from a Python list,
- a $2\times3$ tensor of zeros,
- a $2\times3$ tensor of random values.

For each tensor, print its values, shape, and dtype.

In [ ]:
# TODO: Create the three tensors.


# TODO: Print each tensor, its shape, and its dtype.

## 2. Reshape a batch of images

A grayscale image batch has shape:

$$
(B,C,H,W)=(B,1,28,28).
$$

An MLP needs one vector per image:

$$
(B,1,28,28)\rightarrow(B,784).
$$

Create four random images and flatten every image without combining the batch dimension.

**Checkpoint:** the flattened shape should be `torch.Size([4, 784])`.

In [ ]:
# TODO: Create a tensor with shape (4, 1, 28, 28).


# TODO: Flatten each image while preserving the batch dimension.


print("Original shape:", images.shape)
print("Flattened shape:", flattened.shape)

## 3. Recreate a linear layer using matrix multiplication

For:

$$
Z=XW+b,
$$

use these shapes:

$$
X:(4,784),\qquad W:(784,128),\qquad b:(128,).
$$

Before coding: what should the shape of $Z$ be?

In [ ]:
# TODO: Create X, W, and b with the specified shapes.


# TODO: Calculate Z = X @ W + b.


print("Output shape:", Z.shape)

# Core 2: Autograd

## 4. Calculate gradients automatically

Use:

$$
\widehat y=wx+b,\qquad L=(\widehat y-y)^2,
$$

with:

$$
w=2,\quad b=1,\quad x=3,\quad y=10.
$$

Create trainable tensors for $w$ and $b$, calculate the loss, and call `backward()`.

**Expected values:**

$$
\widehat y=7,\quad L=9,\quad
\frac{\partial L}{\partial w}=-18,\quad
\frac{\partial L}{\partial b}=-6.
$$

In [ ]:
# TODO: Create w and b with gradient tracking enabled.


# TODO: Create x and target.


# TODO: Calculate prediction and squared-error loss.


# TODO: Run backpropagation.


print("Prediction:", prediction.item())
print("Loss:", loss.item())
print("dL/dw:", w.grad.item())
print("dL/db:", b.grad.item())

### Why gradients must be cleared

PyTorch accumulates gradients in `.grad`. A training loop therefore clears previous gradients before the next backward pass.

Clear the gradients of $w$ and $b$ manually. Later, the optimizer will do this through `optimizer.zero_grad()`.

In [ ]:
# TODO: Set the stored gradients of w and b back to zero.


print("w gradient after clearing:", w.grad.item())
print("b gradient after clearing:", b.grad.item())

# Core 3: Build an `nn.Module`

## 5. Define the Fashion-MNIST MLP

Build:

$$
784\rightarrow128\rightarrow64\rightarrow10.
$$

Use:

- `nn.Flatten`
- `nn.Linear`
- `nn.ReLU`
- `nn.Sequential`

`__init__()` creates the layers. `forward()` describes how an input uses them.

In [ ]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()

        # TODO: Create a flatten layer.


        # TODO: Create 784 -> 128 -> 64 -> 10 with ReLU activations.


    def forward(self, x):
        # TODO: Flatten x and pass it through the network.
        pass

## 6. Test the model before training

Create the model on the selected device and pass eight fake images through it.

**Checkpoints:**

- input shape: `(8, 1, 28, 28)`
- output shape: `(8, 10)`
- parameter count: `109,386`

In [ ]:
# TODO: Create FashionMLP and move it to the selected device.


# TODO: Create eight fake images on the same device.


# TODO: Run a forward pass.


trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(model)
print("Input shape:", dummy_images.shape)
print("Output shape:", dummy_logits.shape)
print("Trainable parameters:", f"{trainable_parameters:,}")

## 7. Logits and cross-entropy

The final layer returns ten raw scores called logits. `nn.CrossEntropyLoss` expects:

- logits with shape $(B,10)$,
- integer class labels with shape $(B,)$.

Do not add softmax inside the model. Cross-entropy applies the required log-softmax calculation internally.

In [ ]:
example_logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.2, 0.3, 2.4],
])
example_labels = torch.tensor([0, 2])

# TODO: Create CrossEntropyLoss and calculate the example loss.


# TODO: Convert logits to probabilities only for interpretation.


print("Probabilities:\n", example_probabilities)
print("Loss:", example_loss.item())

# Core 4: Dataset and DataLoader

## 8. Download Fashion-MNIST

This setup is provided so class time stays focused on PyTorch concepts. `ToTensor()` converts images to tensors with shape `(1, 28, 28)` and scales pixels to $[0,1]$.

In [ ]:
DATA_DIR = Path("data")
transform = transforms.ToTensor()

complete_train_dataset = datasets.FashionMNIST(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform,
)
test_dataset = datasets.FashionMNIST(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform,
)

train_dataset, validation_dataset = random_split(
    complete_train_dataset,
    lengths=[55_000, 5_000],
    generator=torch.Generator().manual_seed(SEED),
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Test examples:", len(test_dataset))

## 9. Create mini-batches

Create three `DataLoader` objects with batch size 128.

- Shuffle the training data.
- Do not shuffle validation or test data.

Before coding: why is shuffling useful during training but unnecessary during evaluation?

In [ ]:
BATCH_SIZE = 128

# TODO: Create train_loader with shuffle=True.


# TODO: Create validation_loader and test_loader with shuffle=False.


batch_images, batch_labels = next(iter(train_loader))

print("Image batch shape:", batch_images.shape)
print("Label batch shape:", batch_labels.shape)
print("Number of training batches:", len(train_loader))

In [ ]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

fig, axes = plt.subplots(2, 6, figsize=(10, 4), constrained_layout=True)
for ax, image, label in zip(axes.ravel(), batch_images[:12], batch_labels[:12]):
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(class_names[label.item()])
    ax.axis("off")

plt.show()

# Core 5: One parameter update

## 10. Write the five essential operations

Create a fresh model, cross-entropy loss, and Adam optimizer. Then perform one update:

1. clear gradients,
2. forward pass,
3. calculate loss,
4. backward pass,
5. optimizer step.

Store a copy of the first-layer weights before the update and verify that they changed afterward.

In [ ]:
# TODO: Create a fresh model, loss function, and Adam optimizer.


images, labels = next(iter(train_loader))
images = images.to(device)
labels = labels.to(device)

# TODO: Copy the first linear layer's weights before updating.


# TODO: Perform the five training operations.


# TODO: Read the first-layer weights after the update.


mean_weight_change = (weight_after - weight_before).abs().mean()

print("Logits shape:", logits.shape)
print("Batch loss:", loss.item())
print("Mean absolute weight change:", mean_weight_change.item())

# Core 6: Reusable loops

## 11. Convert one update into one training epoch

Complete the function using the same five operations inside the batch loop.

Accumulate:

- `loss.item() * batch_size`
- the number of correct predictions
- the number of examples

Return average loss and accuracy for the complete epoch.

In [ ]:
def train_one_epoch(model, data_loader, loss_function, optimizer, device):
    # TODO: Put the model in training mode.


    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in data_loader:
        # TODO: Move the batch to the selected device.


        # TODO: Clear gradients, run forward, calculate loss,
        # run backward, and update parameters.


        # TODO: Accumulate loss, correct predictions, and example count.
        pass

    # TODO: Return average loss and accuracy.
    pass

## 12. Write the evaluation loop

Evaluation differs from training:

- use `model.eval()`,
- use `torch.no_grad()`,
- do not call `backward()`,
- do not call `optimizer.step()`.

In [ ]:
def evaluate(model, data_loader, loss_function, device):
    # TODO: Put the model in evaluation mode.


    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    # TODO: Disable gradient tracking.
    with torch.no_grad():
        for images, labels in data_loader:
            # TODO: Move data, run forward, and calculate loss.


            # TODO: Accumulate loss, correct predictions, and count.
            pass

    # TODO: Return average loss and accuracy.
    pass

## 13. Train for three epochs

Three epochs keep the follow-along session practical. More epochs can be run after class.

Create a fresh model so the one-batch experiment does not affect this training run.

In [ ]:
# TODO: Create a fresh model, CrossEntropyLoss, and Adam optimizer.


EPOCHS = 3
history = {
    "train_loss": [],
    "train_accuracy": [],
    "validation_loss": [],
    "validation_accuracy": [],
}

for epoch in range(1, EPOCHS + 1):
    # TODO: Run one training epoch.


    # TODO: Evaluate on the validation loader.


    # TODO: Append all four values to history.


    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} | "
        f"train accuracy {train_accuracy:.3f} | "
        f"validation loss {validation_loss:.4f} | "
        f"validation accuracy {validation_accuracy:.3f}"
    )

## 14. Evaluate the test set

Use the completed `evaluate()` function. This completes the essential PyTorch workflow.

In [ ]:
# TODO: Evaluate the trained model on test_loader.


print("Test loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))

# Core-path checkpoint

At this point, we have covered the minimum complete PyTorch workflow:

$$
\text{tensor}\rightarrow\text{DataLoader}\rightarrow
\text{model}\rightarrow\text{loss}\rightarrow
\text{backward}\rightarrow\text{optimizer step}.
$$

If class time is nearly over, stop here and reconstruct the training loop verbally. The remaining sections are valuable but optional for the first PyTorch session.

# Stretch 1: Plot learning curves

This plotting code is provided. Focus on interpretation:

- Is training loss decreasing?
- Does validation loss follow it?
- Is the train–validation gap growing?

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)

axes[0].plot(epochs, history["train_loss"], marker="o", label="Train")
axes[0].plot(epochs, history["validation_loss"], marker="o", label="Validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy")
axes[0].legend()

axes[1].plot(epochs, history["train_accuracy"], marker="o", label="Train")
axes[1].plot(epochs, history["validation_accuracy"], marker="o", label="Validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].legend()

plt.show()

# Stretch 2: Inspect confident mistakes

This helper code is provided because plotting is not the learning objective. Softmax is used after training to interpret logits as probabilities.

In [ ]:
model.eval()
wrong_images = []
wrong_true = []
wrong_predicted = []
wrong_confidence = []

with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        probabilities = torch.softmax(logits, dim=1).cpu()
        confidence, predictions = probabilities.max(dim=1)
        wrong = predictions != labels

        wrong_images.append(images[wrong])
        wrong_true.append(labels[wrong])
        wrong_predicted.append(predictions[wrong])
        wrong_confidence.append(confidence[wrong])

wrong_images = torch.cat(wrong_images)
wrong_true = torch.cat(wrong_true)
wrong_predicted = torch.cat(wrong_predicted)
wrong_confidence = torch.cat(wrong_confidence)

top_indices = torch.argsort(wrong_confidence, descending=True)[:12]

fig, axes = plt.subplots(3, 4, figsize=(9, 8), constrained_layout=True)
for ax, index in zip(axes.ravel(), top_indices):
    ax.imshow(wrong_images[index].squeeze(), cmap="gray")
    ax.set_title(
        f"true: {class_names[wrong_true[index]]}\n"
        f"pred: {class_names[wrong_predicted[index]]}\n"
        f"p={wrong_confidence[index]:.2f}"
    )
    ax.axis("off")

plt.show()

# Stretch 3: Save and reload parameters

Save `model.state_dict()`. Then create a new `FashionMLP`, load the state dictionary, switch to evaluation mode, and predict one test image.

In [ ]:
MODEL_PATH = Path("fashion_mnist_mlp_state_dict.pth")

# TODO: Save the trained model's state_dict.


# TODO: Create a new FashionMLP on the same device.


# TODO: Load the state_dict using map_location and weights_only=True.


# TODO: Load the weights into the new model and call eval().


sample_image, sample_label = test_dataset[0]

# TODO: Disable gradients and predict the sample image.


print("Actual class:", class_names[sample_label])
print("Predicted class:", class_names[sample_prediction])

# Final retrieval

Explain the following without looking at the earlier cells:

1. What does a tensor's shape tell us?
2. What does `requires_grad=True` do?
3. Why are old gradients cleared?
4. What belongs in `__init__()` and what belongs in `forward()`?
5. Why does `CrossEntropyLoss` receive logits?
6. What is the relationship between `Dataset` and `DataLoader`?
7. What are the five operations in one training update?
8. How does evaluation differ from training?
9. What is stored inside `state_dict`?

## After class

Complete any unfinished stretch sections, increase training to five epochs, and compare CPU and GPU runtime if both are available.